# Tool calling in loops demo


## What is an Agent?

### Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

### The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [9]:
# import necessary libraries 
import os , json 
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI

# load environment variables from .env file
load_dotenv(override=True)


True

In [ ]:
# show normal text in rich format if possible, otherwise show in normal format
def show(text):
    try:
        Console().print(text)
    except Exception as e:
        print(text)

In [11]:
openai = OpenAI(api_key=os.getenv("GITHUB_TOKEN"),base_url=os.getenv("GITHUB_BASE_URL"))

In [12]:
#  two list , one for todos and other for its status (completed or not)
todos = []
completed=[]

In [ ]:
# get todo report function 
def get_todo_report()->str:
    result =""
    for index,todo in enumerate(todos):
        if completed[index]:
            result+=f"Todo #{index+1} :[green][strike] {todo} [/green][/strike]\n"
        else:
            result+=f"Todo #{index+1} : {todo} \n"
    show(result)
    return result

In [14]:
get_todo_report()

''

In [15]:
def create_todos(descriptions: list[str])-> str:
    todos.extend(descriptions)
    completed.extend([False]*len(descriptions))
    return get_todo_report()

In [16]:
# mark complted function
def mark_completed(index:int, completion_notes:str)->str:
    if 1<= index<=len(todos):
        completed[index-1]=True
    else:
        print("No todos at this index")
    Console().print(completion_notes)
    return get_todo_report()

In [17]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1 : Buy groceries 
Todo #2 : Finish extra lab 
Todo #3 : Eat banana

'Todo #1 : Buy groceries \nTodo #2 : Finish extra lab \nTodo #3 : Eat banana \n'

In [18]:
mark_completed(2, "Finished the extra lab successfully")

Finished the extra lab successfully

Todo #1 : Buy groceries 
Todo #2 : Finish extra lab 
Todo #3 : Eat banana

'Todo #1 : Buy groceries \nTodo #2 :[green][strike] Finish extra lab [/green][/strike]\nTodo #3 : Eat banana \n'

In [31]:
create_todos_json = {
    "name":"create_todos",
    "description":"Add new tods with the given descriptions list ",
    "parameters":{
        "type":"object",
        "properties":{
            "descriptions":{
                "type":"array",
                "items":{
                    "type":"string"
                },
                "title":"Descriptions"
            }
        },
        "required":["descriptions"],
        "additionalProperties":False
    }
}

In [33]:
mark_completed_json={
    "name":"mark_completed",
    "description":"Take 1-based index of the todo to mark as completed and completion notes to show",
    "parameters":{
        "type":"object",
        "properties":{
            "index":{
                "type":"integer",
                "title":"Index",
                "description" : "1-based index of the todo to mark as completed"
            },
            "completion_notes":{
                "type":"string",
                "title":"Completion Notes"
            }
        },
        "required":["index","completion_notes"],
        "additionalProperties":False
    }
}

In [34]:
tools = [{"type":"function","function":create_todos_json},{"type":"function","function":mark_completed_json}]

In [43]:
# handle tool calls , takes all tool to calls and return the results after executing the tools
def handle_tool_calls(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results
                       

In [44]:
def loops(messages):
    done = False 
    while not done:
        response = openai.chat.completions.create(model="gpt-4o",messages=messages,tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [45]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loops(messages)  # we gave llm the tools and the problem, now it will use the tools to solve the problem and give us the answer , it will run in loop until it has the answer and then it will give us the answer

# it will show in real time task execution and the final answer at the end

Todo #1 : Estimate the distance between Boston and New York. 
Todo #2 : Calculate the time they meet based on their speeds and departure times. 
Todo #3 : Determine the location where they meet if necessary.

The estimated distance between Boston and New York is approximately 190 miles based on common knowledge.

Todo #1 : Estimate the distance between Boston and New York. 
Todo #2 : Calculate the time they meet based on their speeds and departure times. 
Todo #3 : Determine the location where they meet if necessary.

The two trains meet 1.5 hours after the second train departs, which is at 4:30 pm.

Todo #1 : Estimate the distance between Boston and New York. 
Todo #2 : Calculate the time they meet based on their speeds and departure times. 
Todo #3 : Determine the location where they meet if necessary.

The two trains meet at a point approximately 120 miles from Boston and 70 miles from New York.

Todo #1 : Estimate the distance between Boston and New York. 
Todo #2 : Calculate the time they meet based on their speeds and departure times. 
Todo #3 : Determine the location where they meet if necessary. 

### Solution:

- The trains meet at **4:30 pm**.
- At meeting time, the location is **120 miles from Boston** and **70 miles from New York**.